# Notebook 11 — Phase 2: Git Version Control Setup

This notebook helps prepare your existing project for Git. Git is local version control; creating a local repository does not require GitHub or incur cloud charges.

**Important:** Review `.gitignore` before staging anything. This project contains a large transaction dataset, generated artifacts, and a virtual environment. Do not commit `.venv`, private data, secrets, or large files unintentionally.


In [ ]:
# Cell 1 — Find the project root
from pathlib import Path
import subprocess
import shutil

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Git installed:", shutil.which("git") is not None)
if shutil.which("git") is None:
    raise EnvironmentError("Git was not found. Install Git for Windows, restart VS Code, and rerun this notebook.")


In [ ]:
# Cell 2 — Create or update .gitignore
# These rules avoid staging virtual environments, caches, secrets, and large generated data/model artifacts by default.
gitignore_path = PROJECT_ROOT / ".gitignore"
recommended_rules = [
    "# Python",
    "__pycache__/",
    "*.py[cod]",
    ".venv/",
    "venv/",
    ".pytest_cache/",
    ".ipynb_checkpoints/",
    "",
    "# Secrets and local environment",
    ".env",
    ".env.*",
    "!.env.example",
    "*.pem",
    "*.key",
    "",
    "# Editor / OS",
    ".vscode/",
    ".DS_Store",
    "Thumbs.db",
    "",
    "# Large/private datasets and generated artifacts (track deliberately only if appropriate)",
    "data/raw/",
    "data/processed/",
    "models/*.pkl",
    "models/*.joblib",
    "",
    "# Logs",
    "*.log",
]
existing = gitignore_path.read_text(encoding="utf-8").splitlines() if gitignore_path.exists() else []
merged = existing[:]
for rule in recommended_rules:
    if rule and rule not in merged:
        merged.append(rule)
gitignore_path.write_text("\n".join(merged).rstrip() + "\n", encoding="utf-8")
print("Updated:", gitignore_path)
print(gitignore_path.read_text(encoding="utf-8"))


In [ ]:
# Cell 3 — Initialize Git if needed and configure local identity if already known
def run_git(args, check=True):
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    print("$ git", " ".join(args))
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Git command failed with exit code {result.returncode}")
    return result

git_dir = PROJECT_ROOT / ".git"
if not git_dir.exists():
    run_git(["init"])
else:
    print("Git repository already initialized.")
run_git(["status", "--short", "--branch"])


In [ ]:
# Cell 4 — Review files Git would consider; do not stage blindly
result = run_git(["status", "--short", "--untracked-files=all"], check=False)
print("\nReview the output above.")
print("Check that .venv/, data/raw/, data/processed/, and model pickle files are not listed.")
print("If large/private files appear, update .gitignore before staging.")


In [ ]:
# Cell 5 — Stage only the intended source files and notebooks
# This is a conservative allowlist. Add or remove paths after reviewing your actual project.
candidate_paths = [
    ".gitignore",
    "notebooks",
    "app.py",
    "tests",
    "pytest.ini",
    "requirements.txt",
    "README.md",
]
existing_paths = [p for p in candidate_paths if (PROJECT_ROOT / p).exists()]

print("Files/folders eligible for staging:")
for p in existing_paths:
    print(" -", p)

print("\nStaging these paths only. Large notebooks may still be staged; inspect status before committing.")
if existing_paths:
    run_git(["add", "--", *existing_paths])
run_git(["status", "--short"], check=False)


In [ ]:
# Cell 6 — Verify staged files and create the first commit
# Review the staged list carefully before committing.
run_git(["diff", "--cached", "--stat"], check=False)
staged = run_git(["diff", "--cached", "--name-only"], check=False)
staged_files = [line.strip() for line in staged.stdout.splitlines() if line.strip()]
print("\nStaged file count:", len(staged_files))
print("Staged files:")
for name in staged_files:
    print(" -", name)

blocked_fragments = [".venv/", "data/raw/", "data/processed/", ".env", ".pkl", ".joblib"]
blocked = [name for name in staged_files if any(fragment in name for fragment in blocked_fragments)]
if blocked:
    raise RuntimeError(
        "Potentially private/large files are staged. Unstage and update .gitignore first: "
        + ", ".join(blocked)
    )

print("\nIf the staged list is appropriate, the next cell will create a commit.")


In [ ]:
# Cell 7 — Commit staged project files
# Set your identity for this repository only if Git asks for it.
# Replace the placeholder values with your own preferred Git identity before running if needed.
name_result = run_git(["config", "user.name"], check=False)
email_result = run_git(["config", "user.email"], check=False)

if name_result.returncode != 0 or not name_result.stdout.strip():
    print('Set your local identity with: git config user.name "Your Name"')
if email_result.returncode != 0 or not email_result.stdout.strip():
    print('Set your local identity with: git config user.email "you@example.com"')

if staged_files:
    commit_result = run_git(["commit", "-m", "Set up fraud detection API and tests"], check=False)
    if commit_result.returncode != 0:
        print("\nCommit did not complete. Follow the Git error above, configure identity if required, then rerun this cell.")
else:
    print("No staged files found; nothing to commit.")


In [ ]:
# Cell 8 — Confirm repository status
run_git(["status", "--short", "--branch"], check=False)
print("\nGit setup notebook finished. Confirm the branch/status output is sensible before moving to CI.")


## What success looks like

- Git repository initialized.
- `.gitignore` excludes `.venv`, raw/processed data, secrets, and saved model binaries by default.
- Intended source files and notebooks staged.
- A commit exists (or Git identity was configured and the commit cell is ready to rerun).
- `git status` shows no unexpected staged changes.

## Next step

After local Git setup is complete, the next notebook will prepare GitHub Actions CI. Publishing to GitHub is optional, but a remote repository is needed if you want GitHub-hosted CI to run on pushes and pull requests. Never upload private datasets or credentials.
